# 02 — Feature Engineering & Scheme Detection

Computes roster features, merges game stats, derives style scores, and provides interactive sliders to tune scheme-detection thresholds.


In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.features import build_team_metrics, build_roster_features
from src.scheme_detector import compute_style_scores, apply_labels, build_scheme_widget

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## Load raw data

In [ ]:
stats_df     = pd.read_csv('../data/raw/team_stats/team_stats_raw.csv')
rosters_df   = pd.read_csv('../data/raw/rosters/rosters_raw.csv')
schedules_df = pd.read_csv('../data/raw/schedules/schedules_raw.csv')
print('Stats:', stats_df.shape, '| Rosters:', rosters_df.shape, '| Schedules:', schedules_df.shape)

## Team-level metrics

`yards_per_play`, `pass_rate`, `havoc_rate` are NaN until Sports-Reference CSVs are imported (Step 4 of notebook 01). `points_per_game`, `win_pct`, `ivy_win_pct`, and `avg_margin` come from game data and are always populated.

In [ ]:
metrics_df = build_team_metrics(stats_df)
print('Columns:', metrics_df.columns.tolist())
metrics_df[['school','year','points_per_game','points_allowed_per_game','win_pct','ivy_win_pct']].head(8)

## Roster features

In [ ]:
roster_feats = build_roster_features(rosters_df)
print('Roster feature columns:', roster_feats.columns.tolist())
print('Non-null OL_avg_weight:', roster_feats['OL_avg_weight'].notna().sum(), '/', len(roster_feats))
roster_feats.dropna(subset=['OL_avg_weight']).head(6)

## Merge into master dataset

`stats_df` already contains `ivy_win_pct` from game data, so we use it directly rather than recomputing.

In [ ]:
master = metrics_df.merge(roster_feats, on=['school','year'], how='left')
print('Master shape:', master.shape)
print('Columns:', master.columns.tolist())
master[['school','year','ivy_win_pct','win_pct','points_per_game','OL_avg_weight','SKILL_avg_weight']].head(8)

## Style scores — continuous [0,1] dimensions

In [ ]:
style_df = compute_style_scores(master)
print('Style columns:', [c for c in style_df.columns if c not in ['school','year']])
style_df.head(8)

In [ ]:
style_cols = ['pass_tendency','tempo','spread_factor','explosiveness','aggression','front_heaviness']
available  = [c for c in style_cols if c in style_df.columns and style_df[c].notna().sum() > 0]

if available:
    pivot = style_df.dropna(subset=available, how='all').pivot_table(
        index='school', columns='year', values=available[0]
    )
    fig, ax = plt.subplots(figsize=(14,5))
    sns.heatmap(pivot, cmap='RdYlGn', center=0.5, ax=ax, linewidths=0.3,
                cbar_kws={'label': available[0].replace('_',' ').title()})
    ax.set_title(f'{available[0].replace("_"," ").title()} by School & Year')
    plt.tight_layout()
    plt.savefig('../data/processed/style_heatmap.png', dpi=150)
    plt.show()
else:
    print('Style scores need S-R stat columns (pass_rate, yards_per_play). Import CSVs via notebook 01 Step 4.')

## Interactive Scheme Detection Sliders

In [ ]:
build_scheme_widget(style_df)

## Export master dataset

In [ ]:
labeled = apply_labels(style_df)
master_labeled = master.merge(labeled[['school','year','off_scheme','def_scheme']], on=['school','year'], how='left')
master_labeled.to_csv('../data/processed/master_labeled.csv', index=False)
print('Saved master_labeled.csv —', master_labeled.shape)
master_labeled[['school','year','off_scheme','def_scheme','ivy_win_pct','win_pct','points_per_game']].head(12)